# 📈 BharatBench — Baseline Model: Linear Regression

**Notebook:** `2_Linear_Regression.ipynb`  
**Dataset:** BharatBench (IMDAA reanalysis, 1990–2020, 1.08° resolution, 32×32 grid)  
**Paper:** *Preparing benchmarks for data-driven weather forecasting system over India*  
**Dataset DOI / Download:** [Kaggle — maslab/bharatbench](https://www.kaggle.com/datasets/maslab/bharatbench)  
**Code Repository:** [GitHub — MASLABnitrkl/BharatBench](https://github.com/MASLABnitrkl/BharatBench)

---

## Purpose

This notebook trains and evaluates a **pixel-wise Linear Regression** model as an intermediate baseline between the trivial non-learned forecasts (persistence, climatology) and the deep learning models (CNN, ConvLSTM).

Linear regression serves as an important sanity-check: if a deep learning model cannot convincingly beat linear regression, the added complexity of a neural network is not justified.

### How it works

Each grid point is treated as an independent pixel. The 32 × 32 spatial field is **flattened to a 1024-dimensional vector**, and a single linear map is learned from the input state at time $t_0$ to the target state at lead time $t_0 + \Delta t$:

$$\hat{\mathbf{y}} = \mathbf{W}\mathbf{x} + \mathbf{b}$$

where $\mathbf{x} \in \mathbb{R}^{1024}$ is the flattened normalised input field and $\hat{\mathbf{y}} \in \mathbb{R}^{1024}$ is the predicted output field.

> **Key limitation:** Linear regression assumes a purely linear relationship between atmospheric state and its future evolution. Real atmospheric dynamics are highly non-linear — especially for precipitation — so this model is expected to underperform deep learning on complex variables.

---

## Variables evaluated

| Short name | Variable | Level |
|---|---|---|
| `HGT_prl` | Geopotential Height | 500 hPa |
| `TMP_prl` | Temperature | 850 hPa |
| `TMP_2m` | Temperature | 2 m surface |
| `APCP_sfc` | 6-hourly Accumulated Precipitation | Surface |

---

## Notebook structure

1. [Setup & Imports](#1-setup)
2. [Evaluation Metrics](#2-metrics)
3. [Load Dataset & Splits](#3-dataset)
4. [Normalisation](#4-normalisation)
5. [Linear Regression Model](#5-model)
6. [Experiments & Results](#6-experiments)

---
<a id="1-setup"></a>
## 1. Setup & Imports

All required libraries are standard scientific Python packages. `sklearn` provides the `LinearRegression` solver imported later in Section 5.

> **Environment check:** If any import fails, install the missing package with:
> ```bash
> pip install numpy xarray matplotlib pandas scikit-learn
> ```

In [ ]:
# Libraries 
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

---
<a id="2-metrics"></a>
## 2. Evaluation Metrics

The same three metrics used throughout BharatBench are defined here. These are identical to those in `1_climatology_persistence.ipynb` and are reproduced so each notebook remains self-contained.

### 2.1 Root Mean Square Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{N_{\text{pred}}} \sum_{i} \frac{1}{N_{\text{lat}} N_{\text{lon}}} \sum_{j,k} (f_{i,j,k} - t_{i,j,k})^2}$$

RMSE is the **primary ranking metric** for BharatBench. It penalises large errors more than MAE.

In [ ]:
def compute_rmse(prediction, actual,  mean_dims = ('time', 'latitude', 'longitude')):
  error = prediction - actual
  rmse = np.sqrt(((error)**2 ).mean(mean_dims))
  return rmse

### 2.2 Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{N_{\text{pred}}} \sum_{i} \frac{1}{N_{\text{lat}} N_{\text{lon}}} \sum_{j,k} |f_{i,j,k} - t_{i,j,k}|$$

MAE treats all errors equally and is more robust to outliers than RMSE.

In [ ]:
def compute_mae(prediction, actual, mean_dims = ('time', 'latitude', 'longitude')):
    error = prediction - actual
    mae = np.abs(error).mean(mean_dims)
    return mae

### 2.3 Anomaly Correlation Coefficient (ACC)

$$\text{ACC} = \frac{\sum_{i,j,k} f'_{i,j,k}\, t'_{i,j,k}}{\sqrt{\sum_{i,j,k} f'^2_{i,j,k} \cdot \sum_{i,j,k} t'^2_{i,j,k}}}$$

where primed variables denote departures from the climatological mean.

| ACC range | Interpretation |
|---|---|
| > 0.8 | Highly skillful forecast |
| ≈ 0.6 | Useful forecast |
| ≈ 0.5 | Comparable to climatological mean |
| < 0.3 | Poor skill; limited forecast value |

In [ ]:
def compute_acc(prediction, actual):
    clim = actual.mean('time')
    try:
        t = np.intersect1d(prediction.time, actual.time)
        pred_anomaly = prediction.sel(time=t) - clim
    except AttributeError:
        t = actual.time.values
        pred_anomaly = prediction - clim
    act_anomaly = actual.sel(time=t) - clim
    
    pred_norm = pred_anomaly - pred_anomaly.mean()
    act_norm = act_anomaly - act_anomaly.mean()

    acc = (
            np.sum(pred_norm * act_norm) /
            np.sqrt(
                np.sum(pred_norm ** 2) * np.sum(act_norm ** 2)
            )
    )
    return acc

---
<a id="3-dataset"></a>
## 3. Load Dataset & Splits

The BharatBench dataset is stored as a single NetCDF file. All surface and pressure-level variables for 1990–2020 are contained within it.

> ⚠️ **Update the file path below** to match the location of your downloaded dataset.  
> The dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/maslab/bharatbench).

**Dataset specifications:**

| Property | Value |
|---|---|
| Spatial domain | 5°N – 40°N, 65°E – 100°E |
| Spatial resolution | 1.08° (~32 × 32 grid) |
| Temporal resolution | 6-hourly (00, 06, 12, 18 UTC) |
| Period | 1990–2020 |

In [ ]:
data =  xr.open_dataset(r"G:/IMDAA_Regrid_1.08_1990_2022/IMDAA_merged_1.08_1990_2020.nc")
data

### Dataset splits

The data is divided into three non-overlapping periods using a **continuous temporal split** — rather than a random split — because meteorological variables exhibit strong temporal autocorrelation. A random split would cause data leakage between training and test sets.

| Split | Period | Purpose |
|---|---|---|
| **Training** | 1990–2018 (29 years) | Fit the linear regression model; compute normalisation statistics |
| **Test** | 2019–2020 (2 years) | Final evaluation of forecast skill |

> **Note:** This notebook merges the validation year (2018) into the training set for linear regression, since the model has no hyper-parameters that require validation-based tuning. The deep learning notebooks use a separate 2018 validation split.

In [ ]:
data_train = data.sel(time=slice('1990', '2018'))
data_test = data.sel(time=slice('2019', '2020'))

`test_data` retains the **un-normalised** observations. This copy is kept separately so that after denormalising the model predictions, errors are computed in original physical units (metres, Kelvin, kg/m²).

In [ ]:
test_data = data.sel(time=slice('2019', '2020'))

---
<a id="4-normalisation"></a>
## 4. Normalisation

Linear regression is sensitive to the scale of input features. All variables are standardised using the **training-set mean and standard deviation** computed over the entire training period (all grid points, all times):

$$x_{\text{norm}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

The same $\mu_{\text{train}}$ and $\sigma_{\text{train}}$ are applied to **both** training and test data to prevent information leakage from the test period.

> ⚠️ **Important:** `data_std` stores the per-variable standard deviations computed only on the training split. These values are also used later to **denormalise** the model's predictions back into physical units before computing RMSE, MAE, and ACC.

In [ ]:
data_mean = data_train.mean().load()
data_std = data_train.std().load()

Inspect the standard deviation values. Larger values (e.g., `HGT_prl` ~100 m) reflect variables with greater absolute variability; smaller values (e.g., `TMP_prl` ~5 K) reflect tighter distributions.

In [ ]:
data_std

Apply z-score normalisation to both splits. After this step, each variable has approximately zero mean and unit variance across the training set.

In [ ]:
# Normalize datasets
data_train = (data_train - data_mean) / data_std
data_test = (data_test - data_mean) / data_std

Verify the spatial dimensions of the dataset. The grid is 32 × 32 — the values of `nlat` and `nlon` are used throughout the model code to reshape flattened vectors back into spatial fields.

In [ ]:
_, nlat, nlon = data_train.HGT_prl.shape; nlat, nlon

Inspect the normalised training dataset structure to confirm all variables and coordinates are present.

In [ ]:
data_train

---
<a id="5-model"></a>
## 5. Linear Regression Model

### 5.1 Import scikit-learn

`LinearRegression` from scikit-learn uses the ordinary least squares (OLS) solver. `n_jobs=16` parallelises the computation across CPU cores, which is important because the output dimension (1024 grid points) is large.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

### 5.2 Data preparation helper — `create_training_data`

This function constructs input–output pairs for a given lead time by **shifting** the time series:

- **Input X:** all time steps from $t_0$ up to $t_{\text{end}} - \Delta t$
- **Target y:** all time steps from $t_{\Delta t}$ to $t_{\text{end}}$

Both arrays are **flattened** from shape `(time, lat, lon)` to `(time, lat×lon)` so that `LinearRegression` can treat each grid point's output as a separate regression target.

```
Example for lead_time_h = 12 (3 days at 6-hourly resolution):

Time index:  0   1   2  ...  N-12  N-11  ...  N
X (input):   ●   ●   ●  ...   ●
y (target):                        ●    ...   ●
```

In [ ]:
def create_training_data(da, lead_time_h, return_valid_time=False):
    """Function to split input and output by lead time."""
    X = da.isel(time=slice(0, -lead_time_h))
    y = da.isel(time=slice(lead_time_h, None))
    valid_time = y.time
    if return_valid_time:
        return X.values.reshape(-1, nlat*nlon), y.values.reshape(-1, nlat*nlon), valid_time
    else:
        return X.values.reshape(-1, nlat*nlon), y.values.reshape(-1, nlat*nlon)

### 5.3 Training and evaluation pipeline — `train_lr`

`train_lr` is the main model pipeline. It:

1. Calls `create_training_data` for each input and output variable.
2. Concatenates multi-variable inputs along the feature axis (enabling multi-variable input experiments).
3. Applies optional subsampling (`data_subsample`) to reduce training data volume if needed.
4. Fits a `LinearRegression` model and prints train / test MSE as a diagnostic.
5. **Denormalises** predictions back to physical units using the saved `data_mean` and `data_std`.
6. Wraps the output in an `xr.DataArray` with correct time coordinates, ready for metric computation.

> 💡 **Denormalisation step:** The model learns to predict in normalised space. To compute physically meaningful RMSE (in metres, Kelvin, etc.), predictions are converted back:
> $$x_{\text{pred}} = \hat{y}_{\text{norm}} \times \sigma_{\text{train}} + \mu_{\text{train}}$$

In [ ]:
def train_lr(lead_time_h, input_vars, output_vars, data_subsample=1):
    """Create data, train a linear regression and return the predictions."""
    X_train, y_train, X_test, y_test = [], [], [], []
    for v in input_vars:
        X, y = create_training_data(
            data_train[v],
            lead_time_h
        )

        X_train.append(X)
        if v in output_vars: y_train.append(y)
        X, y, valid_time = create_training_data(data_test[v], lead_time_h, return_valid_time=True)
        X_test.append(X)
        if v in output_vars: y_test.append(y)
    X_train, y_train, X_test, y_test = [np.concatenate(d, 1) for d in [X_train, y_train, X_test, y_test]]
    

    X_train = X_train[::data_subsample]
    y_train = y_train[::data_subsample]
    
 
    lr = LinearRegression(n_jobs=16)
    lr.fit(X_train, y_train)

    mse_train = mean_squared_error(y_train, lr.predict(X_train))
    mse_test = mean_squared_error(y_test, lr.predict(X_test))
    print(f'Train MSE = {mse_train}'); print(f'Test MSE = {mse_test}')
    preds = lr.predict(X_test).reshape((-1, len(output_vars), nlat, nlon))
  

    fcs = []
    for i, v in enumerate(output_vars):
        fc = xr.DataArray(
            preds[:, i] * data_std[v].values + data_mean[v].values,
            dims=['time', 'latitude', 'longitude'],
            coords={
                'time': valid_time,
                'lat': data_train.lat,
                'lon': data_train.lon
            },
            name=v
        )
        fcs.append(fc)
    return xr.merge(fcs), lr

---
<a id="6-experiments"></a>
## 6. Experiments & Results

### 6.1 Variable list

The four target variables evaluated in the BharatBench benchmark:

In [ ]:
var_name = ['HGT_prl', 'TMP_prl', 'TMP_2m', 'APCP_sfc'] # [H500, T850, T2m, TP6h]

### 6.2 Experiment configurations

Each experiment defines a pair `[input_vars, output_vars]`. The active (uncommented) experiments use **single-variable input** — each variable is predicted from itself alone.

The commented-out experiments show **multi-variable input** configurations that can be enabled to explore whether cross-variable information improves skill (e.g., using Z500 + T850 jointly to predict T850).

| Experiment | Input | Output | Status |
|---|---|---|---|
| 1 | `HGT_prl` | `HGT_prl` | ✅ Active |
| 2 | `TMP_prl` | `TMP_prl` | ✅ Active |
| 3 | `HGT_prl` + `TMP_prl` | `HGT_prl` + `TMP_prl` | 💤 Commented out |
| 4 | `APCP_sfc` | `APCP_sfc` | ✅ Active |
| 5 | `HGT_prl` + `TMP_prl` + `APCP_sfc` | `APCP_sfc` | 💤 Commented out |
| 6 | `TMP_2m` | `TMP_2m` | ✅ Active |
| 7 | `HGT_prl` + `TMP_prl` + `TMP_2m` | `TMP_2m` | 💤 Commented out |

> 💡 To run a multi-variable experiment, simply remove the `#` from the corresponding line.

In [ ]:
experiments = [
    [['HGT_prl'], ['HGT_prl']],
    [['TMP_prl'], ['TMP_prl']],
    # [['HGT_prl', 'TMP_prl'], ['HGT_prl', 'TMP_prl']],
    [['APCP_sfc'], ['APCP_sfc']],
    # [['HGT_prl', 'TMP_prl', 'APCP_sfc'], ['APCP_sfc']],
    [['TMP_2m'], ['TMP_2m']],
    # [['HGT_prl', 'TMP_prl', 'TMP_2m'], ['TMP_2m']],
]

### 6.3 Train and evaluate — 3-day lead time

The training loop iterates over all active experiments and:
- Trains a separate `LinearRegression` model for each variable.
- Evaluates on the test set (2019–2020) at a **3-day lead time** (`lead_time = 3*4 = 12` time steps).
- Prints train and test MSE as a training diagnostic.
- Computes RMSE, MAE, and ACC against the un-normalised test observations.
- Collects results into `df_error`.

> ⏱️ **Expected runtime:** Each experiment fits a linear model over ~40,000 training samples and 1,024 output dimensions. With `n_jobs=16`, this typically completes in under a minute per experiment on a standard laptop.

> 🔁 **To evaluate at 5-day lead time**, change `lead_time = 3*4` to `lead_time = 5*4` (= 20 time steps) and re-run this cell.

In [ ]:
data_subsample = 1
lead_time = 3*4
preds = []
models = []
df_error = pd.DataFrame()
for n, (i, o) in enumerate(experiments):
    # print(f'{n}: Input variables = {i}; output variables = {o}')
    var_name = o[0]
    p, m = train_lr(lead_time, input_vars=i, output_vars=o, data_subsample=data_subsample)
    preds.append(p); models.append(m)
    r = compute_rmse(p, test_data).compute()
    m = compute_mae(p, test_data).compute()
    a = compute_acc(p, test_data).compute()
    df_error[var_name ] = pd.DataFrame({var_name : [r[var_name].values, m[var_name].values, a[var_name].values]}, index=['RMSE', 'MAE', 'ACC'])
    
    #print('; '.join([f'{v} = {r[v].values}' for v in r]) + '\n')

### 6.4 Results summary

The table below shows RMSE, MAE, and ACC for each variable at the evaluated lead time. Use these scores as the **linear regression benchmark** that deep learning models (CNN, ConvLSTM) are expected to exceed.

Rows: `RMSE`, `MAE`, `ACC`. Columns: one per variable.

In [ ]:
df_error

---
## Summary & Next Steps

This notebook has established the linear regression baseline for BharatBench at the 3-day and 5-day lead times.

### Interpreting results

- Linear regression captures broad spatial patterns (e.g., Z500 large-scale circulation) but cannot represent non-linear dynamics.
- Performance on `APCP_sfc` (precipitation) is expected to be poor — precipitation has high spatial and temporal variability that linear models cannot represent.
- **Any deep learning model must achieve lower RMSE and higher ACC than these scores to demonstrate genuine skill.**

### Comparing against non-learned baselines

| Model | Z500 RMSE | T850 RMSE | T2m RMSE | TP6h RMSE |
|---|---|---|---|---|
| Persistence (3-day) | — | — | — | — |
| Climatology | — | — | — | — |
| **Linear Regression (3-day)** | — | — | — | — |

> ✏️ Fill in the table above with your computed values to track progress across notebooks.

### Continuing with BharatBench

| Notebook | Description |
|---|---|
| `1_climatology_persistence.ipynb` | Non-learned baselines (persistence, climatology) |
| `3_CNN_ConvLSTM.ipynb` | CNN, ConvLSTM encoder-decoder model |


---
*BharatBench — MAS Lab, NIT Rourkela. Dataset: [Kaggle](https://www.kaggle.com/datasets/maslab/bharatbench) · Code: [GitHub](https://github.com/MASLABnitrkl/BharatBench)*